# Notebook 18 — is Math-Verify a safer judge than string matching?

Notebook 17 established that the string scoring rule undercounts correct
reads. The natural next move is a proper math-equivalence checker:
[HuggingFace Math-Verify](https://github.com/huggingface/Math-Verify) parses
two answers and decides equivalence with SymPy, which is exactly what
`2^3 = 8` vs `2^{3} = 8` needs.

**The answer is that it is excellent as a checker and unsafe as a scorer
here** — and the reason is about FERMAT, not about the tool.

FERMAT injects errors that are frequently a **unit**, a **coefficient**, a
**variable name**, or a **notation detail**. Those are precisely the
differences an equivalence checker exists to normalise away. So a tool that
is *more* mathematically correct can be *less* suitable as the judge on this
benchmark.

What this notebook does:

1. Runs a sanity suite — including both of FERMAT's confirmed injected errors,
   which Math-Verify must **reject**.
2. Demonstrates the `$...$` wrapping gotcha, which silently inverts results.
3. Compares strict / relaxed / Math-Verify scoring on both models.
4. Builds a **disagreement queue** for manual review.

**Decision, recorded up front: the frozen string rule stays the reported
metric.** Math-Verify is used for triage. Nothing here is applied
automatically.

**No GPU, no model, no images.** Runs in a couple of minutes.

In [2]:
# Auth + code access. Everything here is text; no GPU, no images.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
%pip install -q "antlr4-python3-runtime==4.11"
%pip install -q math-verify

sys.path.insert(0, os.path.abspath("repo"))
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import warnings
warnings.filterwarnings("ignore")          # SymPy deprecation noise

import pilot.canonicalize
import pilot.mathverify
import pilot.rescore

assert pilot.canonicalize.latex_parser_available(), "antlr4 pin is broken"
assert pilot.mathverify.math_verify_available(), "math-verify did not install"
print("pilot + math-verify ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pilot (pyproject.toml) ... done
pilot + math-verify ready


## 1. Sanity suite

Five pairs Math-Verify **must accept** (the string rule's false negatives) and
five it **must reject**. Two of the rejects are FERMAT's confirmed injected
errors — items 55 and 273. If a future version ever merges those, Math-Verify
has become unusable here even for triage.

In [3]:
sanity = pilot.mathverify.run_sanity_cases()
print(sanity[["case", "expected", "got", "ok"]].to_string(index=False))
assert sanity["ok"].all(), "a sanity case regressed -- do not trust anything below"
print("\nall sanity cases pass")

                                      case  expected   got   ok
               cosmetic: brace on exponent      True  True True
  equation scaled by a constant (item 208)      True  True True
     LaTeX thin space in a unit (item 280)      True  True True
           spacing inside a tuple (item 9)      True  True True
  answer only vs the whole chain (item 49)      True  True True
       INJECTED ERROR, sign flip (item 55)     False False True
INJECTED ERROR, wrong numerator (item 273)     False False True
                        different variable     False False True
              different exponent (item 94)     False False True
                   plain different numbers     False False True

all sanity cases pass


## 2. The `$...$` gotcha

Math-Verify's `parse()` must be given a **`$`-delimited** string. On bare input
it silently falls back to plain-number extraction and discards the structure.

This is not a curiosity: it produced **every false positive in the first three
runs** of this analysis and made the results look far better than they were.
`pilot.mathverify.mv_parse` wraps for you; never call `parse()` directly.

In [4]:
from math_verify import parse, verify

for span_a, span_b in [("x = 5", "z = 5"), ("6^n + 5^n", "6^n + 5n")]:
    bare = verify(parse(span_a), parse(span_b))
    wrapped = verify(parse(f"${span_a}$"), parse(f"${span_b}$"))
    print(f"{span_a!r:16s} vs {span_b!r:16s}   bare={bare!s:6s} wrapped={wrapped}")
    print(f"    parse({span_a!r})   -> {parse(span_a)}")
    print(f"    parse('${span_a}$') -> {parse(f'${span_a}$')}")
print("\nBare input compares UNEQUAL things as equal. Always wrap.")

'x = 5'          vs 'z = 5'            bare=True   wrapped=False
    parse('x = 5')   -> [5, '5']
    parse('$x = 5$') -> [Eq(x, 5), 'x = 5']
'6^n + 5^n'      vs '6^n + 5n'         bare=True   wrapped=False
    parse('6^n + 5^n')   -> [5, '5']
    parse('$6^n + 5^n$') -> [5**n + 6**n, '6^n + 5^n']

Bare input compares UNEQUAL things as equal. Always wrap.


## 3. Strict vs relaxed vs Math-Verify

Math-Verify is applied to the span **our** extractor produces, not end-to-end.
Running it end-to-end scored *worse* than the string rule (17/30 vs 18/30 on a
probe) because its answer-extraction heuristics differ from ours on multi-step
derivations. Separating extraction from comparison makes a disagreement
attributable to one or the other.

Correctness uses the **majority cluster**, mirroring `majority_cluster`. An
early version asked whether *any* of the K samples matched — a much weaker
criterion that inflated the result from 80.3% to 86.7%.

In [5]:
import ast

import pandas as pd

RUNS = {
    "Qwen2.5-VL-3B": "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv",
    "Pixtral-12B":   "pixtral_perception_full_n300_pixtral-12b_20260809T211028Z.csv",
}

runs, rows, string_correct = {}, [], {}
for name, fname in RUNS.items():
    df = runs[name] = pd.read_csv(f"{RESULTS_DIR}/{fname}")
    strict = pilot.rescore.rescore_run(df, "strict_v1")["transcription_correct"]
    loose = pilot.rescore.rescore_run(df, "final_term_v4")["transcription_correct"]
    string_correct[name] = loose
    mv = [pilot.mathverify.mv_score_item(
              ast.literal_eval(r["all_transcription_samples_raw"]), r["pert_a"])
          for _, r in df.iterrows()]
    mv_ok = pd.Series([m["correct"] for m in mv], index=df.index)
    rows.append({
        "model": name,
        "strict_v1": f"{strict.sum()}/{len(df)} = {strict.mean():.1%}",
        "final_term_v4": f"{loose.sum()}/{len(df)} = {loose.mean():.1%}",
        "math_verify": f"{mv_ok.sum()}/{len(df)} = {mv_ok.mean():.1%}",
        "gt_unparseable": sum(1 for m in mv if not m["gt_parsed"]),
    })
print(pd.DataFrame(rows).to_string(index=False))

KeyboardInterrupt: 

Math-Verify scores highest — and that is exactly what must not be taken at
face value. The next cell is why.

## 4. The disagreement queue

Every item where the string rule and Math-Verify disagree, with both spans, so
a human can decide. **Nothing here is applied.**

`string_wrong_mv_right` is the interesting direction and it contains both:

- **genuine recoveries** — item 40, where our own extractor emitted a parse
  failure and Math-Verify read the answer fine;
- **false positives** — item 108 (`y²/(9/4)` vs `y²/(11/4)`, different
  coefficients, matched on the trailing chain) and item 71 (the ground truth's
  answer is `dy/dx = 1`, not the `π` mentioned in an aside).

On the 2026-08-11 audit roughly **half** were false positives, concentrated on
`has_error=1` rows. That ratio is the finding, and it is why the reported
metric does not move.

In [ ]:
queues = {}
for name, df in runs.items():
    q = queues[name] = pilot.mathverify.disagreement_queue(
        df, string_correct[name], progress=True)
    n_right = (q.direction == "string_wrong_mv_right").sum()
    print(f"\n=== {name}: {len(q)} disagreements "
          f"({n_right} string-wrong/mv-right, {len(q) - n_right} the other way) ===")
    print(f"    of the string-wrong/mv-right rows, "
          f"{int(q[q.direction == 'string_wrong_mv_right'].has_error.sum())} "
          f"are has_error=1 -- the rows where an injected unit or coefficient "
          f"is most likely to have been normalised away")

audit = queues["Qwen2.5-VL-3B"]
audit = audit[audit.direction == "string_wrong_mv_right"].copy()
# Truncate for display only -- the saved CSV keeps the full spans. Without
# this one long derivation pads every column and the table is unreadable.
for col in ("mv_majority_span", "gt_span"):
    audit[col] = audit[col].astype(str).str.replace(r"\s+", " ", regex=True).str[:46]
print(f"\nAUDIT CANDIDATES (Qwen, {len(audit)}) -- review, do not apply:\n")
print(audit[["item", "has_error", "mv_majority_span", "gt_span"]].to_string(index=False))

In [ ]:
# Save the queue so the review can happen outside this runtime.
AUDIT_DIR = f"{PROJECT_DIR}/figures/mathverify_audit"
os.makedirs(AUDIT_DIR, exist_ok=True)
for name, q in queues.items():
    path = f"{AUDIT_DIR}/disagreements_{name.split('-')[0].lower()}.csv"
    q.to_csv(path, index=False)
    print(f"{len(q):3d} rows -> {path}")

## 5. What to carry out of this notebook

- **The reported metric does not change.** `strict_v1` stays the frozen rule;
  Math-Verify is triage.
- **Limitations sentence:** *symbolic equivalence tools are valuable for audit
  but unsafe as automatic scoring on this benchmark, because the injected
  errors are often units, coefficients, or notation details that an
  equivalence checker is designed to erase.*
- **Math-Verify is trustworthy on isolated, `$`-wrapped answer pairs** — it
  passes all ten sanity cases and rejects both injected errors. The problem is
  the surrounding task, not the tool.
- **The queue has ongoing value.** It is how item 40 was found. Re-run it after
  any extractor change and read the new rows.
- **If you ever want an LLM judge**, it belongs *after* this queue, on the rows
  Math-Verify cannot settle — never as a replacement for the frozen rule.